# Entrenamiento híbrido ResNet18 + KAN

Arquitectura para mamografías:

1. **Backbone:** `ResNet18` con pesos ImageNet (`ResNet18_Weights.DEFAULT`).
2. Tras el backbone: vector **512-D** (GAP implícito en el flujo ResNet → KAN).
3. **Cabezal KAN:** p.ej. `KAN([512, 32, 2], grid=3, k=3)` (ver hiperparámetros en el código).
4. **Dos fases:** (1) entrenar sobre todo la KAN con backbone congelado; (2) descongelar `layer4` y afinar conjunto.
5. **Actualizaciones de grid** periódicas para alinear splines con el rango de activaciones.
6. Opcional: **L-BFGS** solo sobre la KAN al final.

**Salidas típicas:** checkpoint híbrido y CSV de historial en `reports/models/`.


In [1]:
# Imports globales, semilla y comprobación de pykan. El filtro de warnings evita ruido de std en tensores pequeños.
from pathlib import Path
from collections import Counter
import copy
import importlib.util
import random
import time
import warnings

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, f1_score, precision_score

warnings.filterwarnings('ignore', message='std\(\): degrees of freedom is <= 0')


<>:22: SyntaxWarning: invalid escape sequence '\('
<>:22: SyntaxWarning: invalid escape sequence '\('
C:\Users\santy\AppData\Local\Temp\ipykernel_23540\2100890761.py:22: SyntaxWarning: invalid escape sequence '\('
  warnings.filterwarnings('ignore', message='std\(\): degrees of freedom is <= 0')


In [ ]:
if not importlib.util.find_spec('kan'):
    raise RuntimeError('KAN no instalado. Ejecuta `%pip install pykan`, reinicia el kernel y corre de nuevo.')

from kan import KAN

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
TRAIN_CSV = ROOT / 'src/data/processed/manifest_train.csv'
TEST_CSV = ROOT / 'src/data/processed/manifest_test.csv'
OUT_DIR = ROOT / 'reports/models'
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'resnet18_kan_hybrid_gap32_GRID_TRUE'
BEST_PATH = OUT_DIR / f'{MODEL_NAME}.pt'
HISTORY_PATH = OUT_DIR / f'{MODEL_NAME}_history.csv'

IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 0

PHASE1_EPOCHS = 6
PHASE2_EPOCHS = 6
EARLY_STOP_PATIENCE = 4

PHASE1_LR = 1e-3
PHASE2_LR_KAN = 5e-4
PHASE2_LR_BACKBONE = 1e-5
WEIGHT_DECAY = 1e-4

GRID_UPDATE_ENABLED = True
GRID_UPDATE_AFTER_EPOCH = 1
GRID_UPDATE_EVERY = 1
GRID_UPDATE_MAX_BATCHES = 8

RUN_LBFGS_FINETUNE = False
LBFGS_STEPS = 5
LBFGS_LR = 0.3
LBFGS_MAX_BATCHES = 4

print('DEVICE:', DEVICE)
print('Best path:', BEST_PATH)


DEVICE: cuda
Best path: G:\Cosas_programacion\Breast Cancer Interpretable-ml\reports\models\resnet18_kan_hybrid_gap32_GRID_TRUE.pt


In [3]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

patients = train_df['patient_id'].dropna().unique()
train_pat, val_pat = train_test_split(patients, test_size=0.2, random_state=SEED)

tr_df = train_df[train_df['patient_id'].isin(train_pat)].copy()
val_df = train_df[train_df['patient_id'].isin(val_pat)].copy()

print('Train:', tr_df.shape, tr_df['label_name'].value_counts().to_dict())
print('Val:', val_df.shape, val_df['label_name'].value_counts().to_dict())
print('Test:', test_df.shape, test_df['label_name'].value_counts().to_dict())


Train: (2315, 10) {'BENIGN': 1383, 'MALIGNANT': 932}
Val: (549, 10) {'BENIGN': 300, 'MALIGNANT': 249}
Test: (422, 10) {'BENIGN': 248, 'MALIGNANT': 174}


In [4]:
class MammographyDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['image_path_local']).convert('RGB')
        x = self.transform(image)
        y = int(row['label'])
        return x, y


train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=8),
    transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.97, 1.03)),
    transforms.ColorJitter(brightness=0.08, contrast=0.12),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_loader = DataLoader(MammographyDataset(tr_df, train_tfms), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
val_loader = DataLoader(MammographyDataset(val_df, eval_tfms), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(MammographyDataset(test_df, eval_tfms), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


In [5]:
class HybridResNetKAN(nn.Module):
    """ResNet-18 + capa de compresión + KAN para clasificación end-to-end.

    Arquitectura (Rama B):
        ResNet-18 backbone (features 512-dim)
        → Linear(512 → 64) + GELU          # bottleneck aprendible
        → KAN([64, 8, 2], grid=3, k=3)     # 528 splines (vs 16384 sin bottleneck)

    Reducir el KAN a [64, 8, 2] hace el backward ~30x más rápido que [512, 32, 2],
    manteniendo interpretabilidad: las 64 dimensiones son combinaciones del embedding.
    La capa Linear actúa como selección/combinación de features antes del KAN.
    """

    def __init__(self, bottleneck_dim=64, kan_hidden=8, kan_grid=3, kan_k=3):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.stem = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3
        self.layer4 = base.layer4
        self.gap = base.avgpool

        # Bottleneck: comprime 512 → bottleneck_dim antes del KAN
        self.bottleneck = nn.Sequential(
            nn.Linear(512, bottleneck_dim),
            nn.GELU(),
        )
        self.kan = KAN(
            width=[bottleneck_dim, kan_hidden, 2],
            grid=kan_grid, k=kan_k,
            auto_save=False, seed=SEED,
        )

    def forward_features(self, x):
        """Devuelve embedding post-bottleneck (entrada al KAN)."""
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        z = self.gap(x).flatten(1)        # [B, 512]
        return self.bottleneck(z)         # [B, bottleneck_dim]

    def forward(self, x):
        z = self.forward_features(x)      # [B, bottleneck_dim]
        return self.kan(z)                # [B, 2]


def freeze_backbone(m):
    for p in m.stem.parameters(): p.requires_grad = False
    for p in m.layer1.parameters(): p.requires_grad = False
    for p in m.layer2.parameters(): p.requires_grad = False
    for p in m.layer3.parameters(): p.requires_grad = False
    for p in m.layer4.parameters(): p.requires_grad = False
    for p in m.gap.parameters(): p.requires_grad = False


def unfreeze_layer4_and_kan(m):
    for p in m.layer4.parameters(): p.requires_grad = True
    for p in m.bottleneck.parameters(): p.requires_grad = True


model = HybridResNetKAN(
    bottleneck_dim=64, kan_hidden=8, kan_grid=3, kan_k=3
).to(DEVICE)

kan_params = sum(p.numel() for p in model.kan.parameters())
bottle_params = sum(p.numel() for p in model.bottleneck.parameters())
total = sum(p.numel() for p in model.parameters())
print(f'HybridResNetKAN (bottleneck=64, kan=[64,8,2]):')
print(f'  Splines KAN:       {64*8 + 8*2} (vs {512*32 + 32*2} sin bottleneck)')
print(f'  Params bottleneck: {bottle_params:,}')
print(f'  Params KAN:        {kan_params:,}')
print(f'  Params total:      {total:,}')


HybridResNetKAN (bottleneck=64, kan=[64,8,2]):
  Splines KAN:       528 (vs 16448 sin bottleneck)
  Params bottleneck: 32,832
  Params KAN:        8,152
  Params total:      11,217,496


In [6]:
counts = Counter(tr_df['label'].tolist())
class_weights = torch.tensor([
    len(tr_df) / (2 * counts[0]),
    len(tr_df) / (2 * counts[1]),
], dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
class_weights


tensor([0.8369, 1.2420], device='cuda:0')

In [7]:
def compute_metrics(y_true, y_pred, y_prob):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_prob = np.asarray(y_prob)
    return {
        'acc': accuracy_score(y_true, y_pred),
        'auc': roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else float('nan'),
        'recall_malignant': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        'precision_malignant': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'f1_malignant': f1_score(y_true, y_pred, pos_label=1, zero_division=0),
    }


def run_epoch(loader, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss = 0.0
    total_seen = 0
    y_true, y_pred, y_prob = [], [], []

    with torch.set_grad_enabled(training):
        for x, y in loader:
            if x.size(0) < 2:
                continue

            x = x.to(DEVICE)
            y = y.to(DEVICE)

            if training:
                optimizer.zero_grad()

            logits = model(x)
            loss = criterion(logits, y)

            if training:
                loss.backward()
                optimizer.step()

            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = logits.argmax(1)

            batch_size = y.size(0)
            total_loss += loss.item() * batch_size
            total_seen += batch_size
            y_true.extend(y.detach().cpu().numpy().tolist())
            y_pred.extend(preds.detach().cpu().numpy().tolist())
            y_prob.extend(probs.detach().cpu().numpy().tolist())

    metrics = compute_metrics(y_true, y_pred, y_prob)
    metrics['loss'] = total_loss / total_seen
    return metrics


def maybe_update_kan_grid(loader, max_batches=GRID_UPDATE_MAX_BATCHES):
    if not GRID_UPDATE_ENABLED:
        return False
    if not hasattr(model.kan, 'update_grid_from_samples'):
        return False

    model.eval()
    features = []
    with torch.no_grad():
        for batch_idx, (x, _) in enumerate(loader):
            x = x.to(DEVICE)
            z = model.forward_features(x)
            features.append(z)
            if batch_idx + 1 >= max_batches:
                break

    if not features:
        return False

    samples = torch.cat(features, dim=0)
    model.kan.update_grid_from_samples(samples)
    return True


def build_phase2_optimizer(model):
    return torch.optim.Adam([
        {'params': model.layer4.parameters(), 'lr': PHASE2_LR_BACKBONE},
        {'params': model.bottleneck.parameters(), 'lr': PHASE2_LR_KAN},
        {'params': model.kan.parameters(), 'lr': PHASE2_LR_KAN},
    ], weight_decay=WEIGHT_DECAY)


In [8]:
# Smoke test corto de forward/backward
freeze_backbone(model)
smoke_optimizer = torch.optim.Adam(model.kan.parameters(), lr=PHASE1_LR)
model.train()
t0 = time.time()
for i, (x, y) in enumerate(train_loader):
    if i >= 2:
        break
    x = x.to(DEVICE)
    y = y.to(DEVICE)
    smoke_optimizer.zero_grad()
    logits = model(x)
    loss = criterion(logits, y)
    loss.backward()
    smoke_optimizer.step()
    print(f'batch {i} loss={loss.item():.4f}')
print('Smoke test seconds:', round(time.time() - t0, 2))


batch 0 loss=0.6932
batch 1 loss=0.6961
Smoke test seconds: 7.39


In [9]:
history = []
best_auc = -1.0
best_state = None
wait = 0

# Phase 1: train only KAN
freeze_backbone(model)
optimizer = torch.optim.Adam(model.kan.parameters(), lr=PHASE1_LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

for epoch in range(1, PHASE1_EPOCHS + 1):
    tr = run_epoch(train_loader, optimizer=optimizer)
    va = run_epoch(val_loader, optimizer=None)
    scheduler.step(va['auc'] if not np.isnan(va['auc']) else 0.0)

    did_grid_update = False
    if GRID_UPDATE_ENABLED and epoch >= GRID_UPDATE_AFTER_EPOCH and epoch % GRID_UPDATE_EVERY == 0:
        did_grid_update = maybe_update_kan_grid(train_loader)

    row = {'phase': 1, 'epoch': epoch, 'grid_update': did_grid_update, **{f'tr_{k}': v for k, v in tr.items()}, **{f'va_{k}': v for k, v in va.items()}}
    history.append(row)
    print(f"P1 E{epoch:02d} | tr_auc={tr['auc']:.4f} va_auc={va['auc']:.4f} va_f1={va['f1_malignant']:.4f} grid_update={did_grid_update}")

    if va['auc'] > best_auc:
        best_auc = va['auc']
        wait = 0
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, BEST_PATH)
    else:
        wait += 1
        if wait >= EARLY_STOP_PATIENCE:
            print('Early stop in phase 1')
            break

# Phase 2: unfreeze layer4 + KAN
model.load_state_dict(best_state)
unfreeze_layer4_and_kan(model)
optimizer = build_phase2_optimizer(model)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
wait = 0

for epoch in range(1, PHASE2_EPOCHS + 1):
    tr = run_epoch(train_loader, optimizer=optimizer)
    va = run_epoch(val_loader, optimizer=None)
    scheduler.step(va['auc'] if not np.isnan(va['auc']) else 0.0)

    did_grid_update = False
    if GRID_UPDATE_ENABLED and epoch >= GRID_UPDATE_AFTER_EPOCH and epoch % GRID_UPDATE_EVERY == 0:
        did_grid_update = maybe_update_kan_grid(train_loader)

    row = {'phase': 2, 'epoch': epoch, 'grid_update': did_grid_update, **{f'tr_{k}': v for k, v in tr.items()}, **{f'va_{k}': v for k, v in va.items()}}
    history.append(row)
    print(f"P2 E{epoch:02d} | tr_auc={tr['auc']:.4f} va_auc={va['auc']:.4f} va_f1={va['f1_malignant']:.4f} grid_update={did_grid_update}")

    if va['auc'] > best_auc:
        best_auc = va['auc']
        wait = 0
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, BEST_PATH)
    else:
        wait += 1
        if wait >= EARLY_STOP_PATIENCE:
            print('Early stop in phase 2')
            break

model.load_state_dict(best_state)
print('Best val AUC:', round(best_auc, 4))
print('Saved:', BEST_PATH)


P1 E01 | tr_auc=0.5227 va_auc=0.6013 va_f1=0.4865 grid_update=True
P1 E02 | tr_auc=0.5935 va_auc=0.6220 va_f1=0.5508 grid_update=True
P1 E03 | tr_auc=0.6198 va_auc=0.6104 va_f1=0.6047 grid_update=True
P1 E04 | tr_auc=0.6389 va_auc=0.6046 va_f1=0.5451 grid_update=True
P1 E05 | tr_auc=0.6353 va_auc=0.5962 va_f1=0.5829 grid_update=True
P1 E06 | tr_auc=0.6598 va_auc=0.5919 va_f1=0.5714 grid_update=True
Early stop in phase 1
P2 E01 | tr_auc=0.6074 va_auc=0.6453 va_f1=0.5360 grid_update=True
P2 E02 | tr_auc=0.7035 va_auc=0.6727 va_f1=0.5195 grid_update=True
P2 E03 | tr_auc=0.7253 va_auc=0.6680 va_f1=0.5720 grid_update=True
P2 E04 | tr_auc=0.7574 va_auc=0.6664 va_f1=0.6035 grid_update=True
P2 E05 | tr_auc=0.7670 va_auc=0.6737 va_f1=0.6294 grid_update=True
lstsq failed


UnboundLocalError: cannot access local variable 'coef' where it is not associated with a value

In [ ]:
RUN_LBFGS_FINETUNE = True

In [ ]:
# Etapa opcional: ajuste fino de la KAN con L-BFGS
if RUN_LBFGS_FINETUNE:
    freeze_backbone(model)
    lbfgs = torch.optim.LBFGS(model.kan.parameters(), lr=LBFGS_LR, max_iter=20, history_size=20)

    cache_batches = []
    for batch_idx, (x, y) in enumerate(train_loader):
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        cache_batches.append((x, y))
        if batch_idx + 1 >= LBFGS_MAX_BATCHES:
            break

    def closure():
        lbfgs.zero_grad()
        total = 0.0
        for x, y in cache_batches:
            logits = model(x)
            loss = criterion(logits, y)
            total = total + loss
        total.backward()
        return total

    for step in range(LBFGS_STEPS):
        loss = lbfgs.step(closure)
        print(f'LBFGS step {step + 1:02d} | loss={float(loss):.4f}')

    va = run_epoch(val_loader, optimizer=None)
    print('Val after LBFGS:', va)

    if va['auc'] > best_auc:
        best_auc = va['auc']
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, BEST_PATH)
        print('LBFGS improved the checkpoint.')
else:
    print('RUN_LBFGS_FINETUNE=False. Se omite esta etapa.')


C:\Users\santy\AppData\Local\Temp\ipykernel_23420\3792305744.py:26: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  print(f'LBFGS step {step + 1:02d} | loss={float(loss):.4f}')


LBFGS step 01 | loss=1.8207
LBFGS step 02 | loss=0.4871
LBFGS step 03 | loss=0.0011
LBFGS step 04 | loss=0.0000
LBFGS step 05 | loss=0.0000
Val after LBFGS: {'acc': 0.604735883424408, 'auc': 0.645053547523427, 'recall_malignant': 0.357429718875502, 'precision_malignant': 0.6095890410958904, 'f1_malignant': 0.4506329113924051, 'loss': 60.084983047457555}


In [ ]:
hist_df = pd.DataFrame(history)
hist_df.to_csv(HISTORY_PATH, index=False)
display(hist_df.tail(10))
print('History saved:', HISTORY_PATH)


,phase,epoch,grid_update,tr_acc,tr_auc,tr_recall_malignant,tr_precision_malignant,tr_f1_malignant,tr_loss,va_acc,va_auc,va_recall_malignant,va_precision_malignant,va_f1_malignant,va_loss
2,1,3,False,0.584201,0.600856,0.512959,0.483707,0.497904,0.679236,0.595628,0.602831,0.481928,0.563380,0.519481,0.684817
3,1,4,False,0.587240,0.625450,0.530744,0.488095,0.508527,0.670713,0.535519,0.600154,0.722892,0.491803,0.585366,0.677818
4,1,5,False,0.606337,0.648994,0.615551,0.508475,0.556913,0.660738,0.613843,0.600863,0.602410,0.570342,0.585938,0.681600
5,1,6,False,0.617188,0.652637,0.621767,0.520758,0.566798,0.658166,0.593807,0.601439,0.429719,0.569149,0.489703,0.693016
6,2,1,False,0.587240,0.629367,0.538213,0.489237,0.512558,0.666907,0.570128,0.667075,0.228916,0.564356,0.325714,0.732756
7,2,2,False,0.654514,0.708568,0.741658,0.553414,0.633855,0.616188,0.650273,0.677369,0.598394,0.618257,0.608163,0.659615
8,2,3,False,0.690104,0.756850,0.784250,0.585818,0.670664,0.572382,0.641166,0.681506,0.670683,0.592199,0.629002,0.652919
9,2,4,False,0.689236,0.755045,0.811015,0.581269,0.677187,0.569760,0.626594,0.677791,0.742972,0.567485,0.643478,0.632556
10,2,5,False,0.694444,0.776046,0.829925,0.585421,0.686554,0.554607,0.624772,0.671386,0.726908,0.567398,0.637324,0.654698
11,2,6,False,0.677083,0.760959,0.814455,0.568953,0.669920,0.567850,0.613843,0.677892,0.795181,0.551532,0.651316,0.638928


History saved: G:\Cosas_programacion\Breast Cancer Interpretable-ml\reports\models\resnet18_kan_hybrid_gap32_history.csv


In [ ]:
model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE))
test_metrics = run_epoch(test_loader, optimizer=None)
print('Test metrics:', test_metrics)


Test metrics: {'acc': 0.5781990521327014, 'auc': 0.6356368186874304, 'recall_malignant': 0.6379310344827587, 'precision_malignant': 0.4911504424778761, 'f1_malignant': 0.555, 'loss': 0.6720914851997701}
